# `delay_twostage_antigravity` run notebook

Trains and evaluates the `delay_twostage_antigravity` arm on its own, with real output kept in the cells below (not lost to the terminal).

**Technique:** Hurdle / two-stage: baseline lookup -> binary delay classifier -> delay-magnitude regressor.

**Question this arm answers:** *separate "is there a delay" from "how large is it"*

Assumes its dependencies (if any -- see `exploration/pipelines_overview.csv` `parents` column) are already trained under the same protocol.

In [ ]:
from __future__ import annotations

import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.py").exists() and (ROOT.parent / "config.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.py").exists(), f"can't locate exploration/ root from {Path.cwd()}"
sys.path.insert(0, str(ROOT))
import config

ARM = "delay_twostage_antigravity"
SPLIT = config.DEFAULT_SPLIT
FEATURESET = config.DEFAULT_FEATURESET
POLICY = config.DEFAULT_OUTLIER_POLICY
SEED = 42
BASE_ARGS = ["--split", SPLIT, "--featureset", FEATURESET,
             "--outlier-policy", POLICY, "--seed", str(SEED)]
print(f"arm={ARM}  protocol: split={SPLIT} featureset={FEATURESET} outlier_policy={POLICY} seed={SEED}")


def run(cmd: "list[str]") -> None:
    """Run a stage script and stream its stdout/stderr into THIS cell's output.

    subprocess.call() with no stdout= argument inherits the OS-level file
    descriptor, bypassing Jupyter's stdout capture. Piping it through Python's
    print() puts it through sys.stdout instead, which Jupyter does capture.
    """
    print("\n" + "=" * 78)
    print("$ " + " ".join(cmd))
    print("=" * 78, flush=True)
    t0 = time.time()
    proc = subprocess.Popen([sys.executable] + cmd, cwd=str(ROOT),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"[{time.time() - t0:.1f}s, rc={rc}]")
    if rc != 0:
        raise SystemExit(f"HALT: `{' '.join(cmd)}` failed with rc={rc}.")


## Train

In [ ]:
run(["pipeline_delay_twostage_antigravity/train.py"] + BASE_ARGS + ["--force"])

## Evaluate (val, then test)

In [ ]:
for part in ("val", "test"):
    run(["-m", "pipeline_delay_twostage_antigravity.evaluate"] + BASE_ARGS + ["--part", part, "--force"])

## Summary

In [ ]:
import json
from common.bootstrap import run_dir

proto = config.protocol_key(SPLIT, FEATURESET, POLICY)
d = run_dir(config.ARTIFACTS / "runs", ARM, protocol=proto, seed=SEED)
val = json.loads((d / "eval_val.json").read_text())
test = json.loads((d / "eval_test.json").read_text())
print(f"{ARM:<28s} val_rmse={val['rmse']:>8.2f}  test_rmse={test['rmse']:>8.2f}")
